# Hugh behaviour review

Inspect saved responses beside their colour swatches, exact setup and attributed judgments.
**Peanut judges first.** The latest three responses await both evaluators.

**Run All** reads `.local/behaviour.sqlite`; it makes no model calls or database writes.
These saved minis predate the removal of the “finer choice” cue. Their original wording stays visible.

In [1]:
import json
import re
import sqlite3
from contextlib import closing
from html import escape
from pathlib import Path
from IPython.display import HTML, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/huemiliator").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the Huemiliator repository.")
DB = ROOT / ".local/behaviour.sqlite"
RUN_ID = None  # None selects the latest saved mini; use a run ID from the table below.

with closing(sqlite3.connect(DB.as_uri() + "?mode=ro", uri=True)) as conn:
    conn.row_factory = sqlite3.Row
    runs = [dict(r) for r in conn.execute("""
        SELECT run_id, COUNT(*) AS outputs, MIN(id) AS first_id, MAX(id) AS last_id,
               model, reasoning_effort, instructions_version, bank_version
        FROM eval_outputs
        GROUP BY run_id, model, reasoning_effort, instructions_version, bank_version
        ORDER BY MAX(id) DESC
    """)]
    if not runs:
        raise ValueError("Import a saved mini using the behaviour records guide first.")
    selected_run = RUN_ID or runs[0]["run_id"]
    outputs = [dict(r) for r in conn.execute(
        "SELECT * FROM eval_outputs WHERE run_id=? ORDER BY id", (selected_run,))]
    judgments = [dict(r) for r in conn.execute("""
        SELECT j.* FROM latest_judgments j
        JOIN eval_outputs o ON o.id=j.output_id WHERE o.run_id=? ORDER BY j.id
    """, (selected_run,))]
if not outputs:
    raise ValueError("That run ID has no saved outputs.")
print(f"Loaded {len(outputs)} responses from {selected_run} in read-only mode.")

Loaded 3 responses from 20260920T012801Z-cues in read-only mode.


## Saved minis

Versions below describe what actually ran. Mechanical checks and behavioural judgments are separate.

In [2]:
def table(headers, rows):
    th = "".join(f'<th style="text-align:left;padding:8px">{escape(h)}</th>' for h in headers)
    body = "".join("<tr>" + "".join(
        f'<td style="padding:8px;border-top:1px solid #ddd">{escape(str(v))}</td>'
        for v in row) + "</tr>" for row in rows)
    return HTML(f'<table style="border-collapse:collapse"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>')

display(table(["Run", "Rows", "Model / reasoning", "Instructions / bank"], [
    [r["run_id"], f'{r["first_id"]}–{r["last_id"]}',
     f'{r["model"]} / {r["reasoning_effort"]}',
     f'{r["instructions_version"]} / {r["bank_version"]}'] for r in runs]))

Run,Rows,Model / reasoning,Instructions / bank
20260920T012801Z-cues,4–6,gpt-5.6-luna / medium,1.3.0 / 0.5.0
20260920T010230Z-316de2,1–3,gpt-5.6-luna / medium,1.2.0 / 0.4.0


## Response review

Each card shows the original response. Expand the record details for mechanical checks,
source hash and a link to the original JSON. The review rows come from the latest judgment **per evaluator**.

In [3]:
def show_response(row):
    record = json.loads(row["record_json"])
    chips = []
    for swatch in record["request"]["display_swatches"]:
        colour = swatch["hex"]
        if not re.fullmatch(r"#[0-9a-fA-F]{6}", colour):
            raise ValueError("Invalid stored swatch colour.")
        chips.append(f'<div style="display:flex;align-items:center;gap:12px">'
                     f'<span style="display:inline-block;width:48px;height:48px;background:{colour};border:1px solid #ddd"></span>'
                     f'<span>{escape(swatch["label"])}</span></div>')
    current = {j["evaluator"]: j for j in judgments if j["output_id"] == row["id"]}
    reviews = []
    for evaluator in dict.fromkeys(["Peanut", "primary assistant", *current]):
        judgment = current.get(evaluator)
        verdict = judgment["verdict"].upper() if judgment else "pending"
        note = judgment["note"] if judgment else "Awaiting judgment"
        source = judgment["source"] if judgment else ""
        reviews.append(f'<p><strong>{escape(evaluator)}: {verdict}</strong><br>'
                       f'<span style="white-space:pre-wrap">{escape(note)}</span>'
                       f'<br><small>{escape(source)}</small></p>')
    visible = row["response_text"]
    if visible is None:
        visible = record["api_response"]["output_text"]
    details = json.dumps(record["mechanical_checks"], ensure_ascii=False, indent=2)
    card = f"""<article style="max-width:850px;padding:20px;margin:12px 0 24px;border:1px solid #d9d2ca;border-radius:8px;background:#fffdfa;color:#242424;font-family:system-ui,sans-serif">
      <h3 style="margin-top:0">Record {row['id']} · {escape(row['case_id'])}</h3>
      <div style="display:flex;gap:60px;flex-wrap:wrap">{''.join(chips)}</div>
      <p style="white-space:pre-wrap;font-size:17px;line-height:1.65">{escape(visible)}</p>
      <div style="border-top:1px solid #ded8d1">{''.join(reviews)}</div>
      <details><summary>Record details</summary>
        <p>{escape(row['source_path'])}</p><small>SHA-256: {row['source_sha256']}</small>
        <pre style="white-space:pre-wrap">{escape(details)}</pre>
        <p><a href="../../{escape(row['source_path'], quote=True)}">Open original JSON</a></p>
      </details></article>"""
    display(HTML(card))

for row in outputs:
    show_response(row)

## Evidence and recording

- [Behaviour records guide](../../docs/runtime/BEHAVIOUR_RECORDS.md): schema, imports and attributed judgment commands.
- [Construction research](../../docs/research/220_RESPONSE_CONSTRUCTION.md): the two minis and their limitations.
- [Original mini](../../.local/behaviour-mini-evals/20260920T010230Z-316de2/README.md) and [cue mini](../../.local/behaviour-mini-evals/20260920T012801Z-cues/README.md): original requests, responses and source ledgers.

The database retains the original JSON bytes and the full judgment history. Revisions append a new judgment; the notebook refreshes its displayed verdict. Timed behaviour pulses remain in staging.